# Telecom Customer Churn & Retention
## 02 — SQL Business Analysis

The analysis focuses on:
- overall churn performance;
- contract and tenure risk;
- service and billing patterns;
- customer demographics;
- customer lifetime value (CLTV);
- historical reasons customers left.


In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 1. Load the cleaned data



In [2]:
candidate_paths = [
    Path("../data/processed/telco_churn_cleaned.csv"),
    Path("data/processed/telco_churn_cleaned.csv"),
    Path("telco_churn_cleaned.csv"),
    Path("/mnt/data/processed/telco_churn_cleaned.csv"),
]

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "telco_churn_cleaned.csv was not found. "
        "Run 01_data_preparation.ipynb first."
    )

df = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")


Loaded: ../data/processed/telco_churn_cleaned.csv
Rows: 7,043
Columns: 33


## 2. Create an in-memory SQLite database

The cleaned DataFrame is loaded into SQLite so the rest of this notebook can use standard SQL queries.


In [3]:
conn = sqlite3.connect(":memory:")

df.to_sql(
    "telco_churn",
    conn,
    if_exists="replace",
    index=False
)

print("SQLite table created: telco_churn")


SQLite table created: telco_churn


In [4]:
def run_query(query):
    return pd.read_sql_query(query, conn)


## 3. Overall churn KPIs

In [5]:
query = """SELECT
    COUNT(*) AS total_customers,
    SUM("Churn Value") AS churned_customers,
    COUNT(*) - SUM("Churn Value") AS retained_customers,
    ROUND(100.0 * SUM("Churn Value") / COUNT(*), 2) AS churn_rate_pct,
    ROUND(AVG("Monthly Charges"), 2) AS avg_monthly_charges,
    ROUND(AVG("Total Charges"), 2) AS avg_total_charges,
    ROUND(AVG(CLTV), 2) AS avg_cltv
FROM telco_churn;"""

run_query(query)


,total_customers,churned_customers,retained_customers,churn_rate_pct,avg_monthly_charges,avg_total_charges,avg_cltv
0,7043,1869,5174,26.54,64.76,"2,279.73","4,400.30"


## 4. Churn by contract type

In [6]:
query = """SELECT
    Contract,
    COUNT(*) AS customers,
    SUM("Churn Value") AS churned_customers,
    ROUND(100.0 * SUM("Churn Value") / COUNT(*), 2) AS churn_rate_pct,
    ROUND(AVG("Monthly Charges"), 2) AS avg_monthly_charges,
    ROUND(AVG(CLTV), 2) AS avg_cltv
FROM telco_churn
GROUP BY Contract
ORDER BY churn_rate_pct DESC;"""

run_query(query)


,Contract,customers,churned_customers,churn_rate_pct,avg_monthly_charges,avg_cltv
0,Month-to-month,3875,1655,42.71,66.40,"4,136.71"
1,One year,1473,166,11.27,65.05,"4,529.96"
2,Two year,1695,48,2.83,60.77,"4,890.21"


## 5. Churn by tenure group

In [7]:
query = """WITH tenure_groups AS (
    SELECT
        *,
        CASE
            WHEN "Tenure Months" <= 6 THEN '0-6 months'
            WHEN "Tenure Months" <= 12 THEN '7-12 months'
            WHEN "Tenure Months" <= 24 THEN '13-24 months'
            WHEN "Tenure Months" <= 48 THEN '25-48 months'
            ELSE '49+ months'
        END AS tenure_group
    FROM telco_churn
)
SELECT
    tenure_group,
    COUNT(*) AS customers,
    SUM("Churn Value") AS churned_customers,
    ROUND(100.0 * SUM("Churn Value") / COUNT(*), 2) AS churn_rate_pct,
    ROUND(AVG("Monthly Charges"), 2) AS avg_monthly_charges,
    ROUND(AVG(CLTV), 2) AS avg_cltv
FROM tenure_groups
GROUP BY tenure_group
ORDER BY
    CASE tenure_group
        WHEN '0-6 months' THEN 1
        WHEN '7-12 months' THEN 2
        WHEN '13-24 months' THEN 3
        WHEN '25-48 months' THEN 4
        WHEN '49+ months' THEN 5
    END;"""

run_query(query)


,tenure_group,customers,churned_customers,churn_rate_pct,avg_monthly_charges,avg_cltv
0,0-6 months,1481,784,52.94,54.74,"4,035.41"
1,7-12 months,705,253,35.89,58.95,"4,038.13"
2,13-24 months,1024,294,28.71,61.36,"4,015.20"
3,25-48 months,1594,325,20.39,65.93,"3,991.46"
4,49+ months,2239,213,9.51,73.95,"5,222.87"


## 6. Churn by internet service

In [8]:
query = """SELECT
    "Internet Service",
    COUNT(*) AS customers,
    SUM("Churn Value") AS churned_customers,
    ROUND(100.0 * SUM("Churn Value") / COUNT(*), 2) AS churn_rate_pct,
    ROUND(AVG("Monthly Charges"), 2) AS avg_monthly_charges
FROM telco_churn
GROUP BY "Internet Service"
ORDER BY churn_rate_pct DESC;"""

run_query(query)


,Internet Service,customers,churned_customers,churn_rate_pct,avg_monthly_charges
0,Fiber optic,3096,1297,41.89,91.50
1,DSL,2421,459,18.96,58.10
2,No,1526,113,7.40,21.08


## 7. Churn by payment method

In [9]:
query = """SELECT
    "Payment Method",
    COUNT(*) AS customers,
    SUM("Churn Value") AS churned_customers,
    ROUND(100.0 * SUM("Churn Value") / COUNT(*), 2) AS churn_rate_pct,
    ROUND(AVG("Monthly Charges"), 2) AS avg_monthly_charges
FROM telco_churn
GROUP BY "Payment Method"
ORDER BY churn_rate_pct DESC;"""

run_query(query)


,Payment Method,customers,churned_customers,churn_rate_pct,avg_monthly_charges
0,Electronic check,2365,1071,45.29,76.26
1,Mailed check,1612,308,19.11,43.92
2,Bank transfer (automatic),1544,258,16.71,67.19
3,Credit card (automatic),1522,232,15.24,66.51


## 8. Churned vs retained customer profile

In [10]:
query = """SELECT
    CASE
        WHEN "Churn Value" = 1 THEN 'Churned'
        ELSE 'Retained'
    END AS customer_status,
    COUNT(*) AS customers,
    ROUND(AVG("Monthly Charges"), 2) AS avg_monthly_charges,
    ROUND(AVG("Total Charges"), 2) AS avg_total_charges,
    ROUND(AVG("Tenure Months"), 2) AS avg_tenure_months,
    ROUND(AVG(CLTV), 2) AS avg_cltv
FROM telco_churn
GROUP BY "Churn Value"
ORDER BY "Churn Value" DESC;"""

run_query(query)


,customer_status,customers,avg_monthly_charges,avg_total_charges,avg_tenure_months,avg_cltv
0,Churned,1869,74.44,"1,531.80",17.98,"4,149.41"
1,Retained,5174,61.27,"2,549.91",37.57,"4,490.92"


## 9. Churn by paperless billing

In [11]:
query = """SELECT
    "Paperless Billing",
    COUNT(*) AS customers,
    SUM("Churn Value") AS churned_customers,
    ROUND(100.0 * SUM("Churn Value") / COUNT(*), 2) AS churn_rate_pct
FROM telco_churn
GROUP BY "Paperless Billing"
ORDER BY churn_rate_pct DESC;"""

run_query(query)


,Paperless Billing,customers,churned_customers,churn_rate_pct
0,Yes,4171,1400,33.57
1,No,2872,469,16.33


## 10. Demographic churn segments

In [12]:
query = """SELECT
    "Senior Citizen",
    Partner,
    Dependents,
    COUNT(*) AS customers,
    SUM("Churn Value") AS churned_customers,
    ROUND(100.0 * SUM("Churn Value") / COUNT(*), 2) AS churn_rate_pct,
    ROUND(AVG(CLTV), 2) AS avg_cltv
FROM telco_churn
GROUP BY "Senior Citizen", Partner, Dependents
HAVING COUNT(*) >= 30
ORDER BY churn_rate_pct DESC;"""

run_query(query)


,Senior Citizen,Partner,Dependents,customers,churned_customers,churn_rate_pct,avg_cltv
0,Yes,No,No,558,273,48.92,"4,229.93"
1,Yes,Yes,No,511,191,37.38,"4,532.82"
2,No,No,No,2781,877,31.54,"4,245.97"
3,No,Yes,No,1566,422,26.95,"4,561.25"
4,No,No,Yes,291,45,15.46,"4,231.97"
5,Yes,Yes,Yes,62,7,11.29,"4,696.63"
6,No,Yes,Yes,1263,49,3.88,"4,587.01"


## 11. Churn by service feature

In [13]:
query = """WITH service_long AS (
    SELECT 'Online Security' AS service, "Online Security" AS service_status, "Churn Value" AS churn_value
    FROM telco_churn

    UNION ALL
    SELECT 'Online Backup', "Online Backup", "Churn Value"
    FROM telco_churn

    UNION ALL
    SELECT 'Device Protection', "Device Protection", "Churn Value"
    FROM telco_churn

    UNION ALL
    SELECT 'Tech Support', "Tech Support", "Churn Value"
    FROM telco_churn

    UNION ALL
    SELECT 'Streaming TV', "Streaming TV", "Churn Value"
    FROM telco_churn

    UNION ALL
    SELECT 'Streaming Movies', "Streaming Movies", "Churn Value"
    FROM telco_churn
)
SELECT
    service,
    service_status,
    COUNT(*) AS customers,
    SUM(churn_value) AS churned_customers,
    ROUND(100.0 * SUM(churn_value) / COUNT(*), 2) AS churn_rate_pct
FROM service_long
GROUP BY service, service_status
ORDER BY service, churn_rate_pct DESC;"""

run_query(query)


,service,service_status,customers,churned_customers,churn_rate_pct
0,Device Protection,No,3095,1211,39.13
1,Device Protection,Yes,2422,545,22.50
2,Device Protection,No internet service,1526,113,7.40
3,Online Backup,No,3088,1233,39.93
4,Online Backup,Yes,2429,523,21.53
5,Online Backup,No internet service,1526,113,7.40
6,Online Security,No,3498,1461,41.77
7,Online Security,Yes,2019,295,14.61
8,Online Security,No internet service,1526,113,7.40
9,Streaming Movies,No,2785,938,33.68


## 12. Churn by CLTV quartile

In [14]:
query = """WITH cltv_ranked AS (
    SELECT
        *,
        NTILE(4) OVER (ORDER BY CLTV) AS cltv_quartile
    FROM telco_churn
)
SELECT
    cltv_quartile,
    COUNT(*) AS customers,
    MIN(CLTV) AS min_cltv,
    MAX(CLTV) AS max_cltv,
    ROUND(AVG(CLTV), 2) AS avg_cltv,
    SUM("Churn Value") AS churned_customers,
    ROUND(100.0 * SUM("Churn Value") / COUNT(*), 2) AS churn_rate_pct
FROM cltv_ranked
GROUP BY cltv_quartile
ORDER BY cltv_quartile;"""

run_query(query)


,cltv_quartile,customers,min_cltv,max_cltv,avg_cltv,churned_customers,churn_rate_pct
0,1,1761,2003,3469,"2,742.81",606,34.41
1,2,1761,3469,4527,"4,079.12",473,26.86
2,3,1761,4528,5381,"4,960.00",425,24.13
3,4,1760,5381,6500,"5,820.06",365,20.74


## 13. Most common churn reasons

In [15]:
query = """SELECT
    "Churn Reason",
    COUNT(*) AS churned_customers,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM telco_churn WHERE "Churn Value" = 1),
        2
    ) AS pct_of_churned_customers
FROM telco_churn
WHERE "Churn Value" = 1
  AND "Churn Reason" IS NOT NULL
GROUP BY "Churn Reason"
ORDER BY churned_customers DESC;"""

run_query(query)


,Churn Reason,churned_customers,pct_of_churned_customers
0,Attitude of support person,192,10.27
1,Competitor offered higher download speeds,189,10.11
2,Competitor offered more data,162,8.67
3,Don't know,154,8.24
4,Competitor made better offer,140,7.49
5,Attitude of service provider,135,7.22
6,Competitor had better devices,130,6.96
7,Network reliability,103,5.51
8,Product dissatisfaction,102,5.46
9,Price too high,98,5.24


## 14. Churn reason categories

In [16]:
query = """WITH categorized_reasons AS (
    SELECT
        CASE
            WHEN "Churn Reason" LIKE 'Competitor%' THEN 'Competitor'
            WHEN "Churn Reason" IN (
                'Attitude of support person',
                'Attitude of service provider',
                'Lack of self-service on Website'
            ) THEN 'Customer Service'
            WHEN "Churn Reason" IN (
                'Network reliability',
                'Product dissatisfaction',
                'Service dissatisfaction',
                'Limited range of services',
                'Poor expertise of online support',
                'Poor expertise of phone support'
            ) THEN 'Product / Service'
            WHEN "Churn Reason" IN (
                'Price too high',
                'Extra data charges',
                'Long distance charges'
            ) THEN 'Price / Charges'
            WHEN "Churn Reason" IN (
                'Moved',
                'Deceased'
            ) THEN 'External / Unavoidable'
            ELSE 'Other'
        END AS churn_reason_category
    FROM telco_churn
    WHERE "Churn Value" = 1
      AND "Churn Reason" IS NOT NULL
)
SELECT
    churn_reason_category,
    COUNT(*) AS churned_customers,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM categorized_reasons),
        2
    ) AS pct_of_churned_customers
FROM categorized_reasons
GROUP BY churn_reason_category
ORDER BY churned_customers DESC;"""

run_query(query)


,churn_reason_category,churned_customers,pct_of_churned_customers
0,Competitor,621,33.23
1,Customer Service,415,22.20
2,Product / Service,377,20.17
3,Price / Charges,199,10.65
4,Other,198,10.59
5,External / Unavoidable,59,3.16


## 15. Highest-CLTV historical churned customers

In [17]:
query = """SELECT
    CustomerID,
    City,
    Contract,
    "Tenure Months",
    "Monthly Charges",
    "Total Charges",
    CLTV,
    "Churn Reason"
FROM telco_churn
WHERE "Churn Value" = 1
ORDER BY CLTV DESC
LIMIT 20;"""

run_query(query)


,CustomerID,City,Contract,Tenure Months,Monthly Charges,Total Charges,CLTV,Churn Reason
0,1043-YCUTE,Hayward,Two year,56,25.15,"1,327.15",6484,Service dissatisfaction
1,1323-OOEPC,Prather,Month-to-month,53,98.40,"5,149.50",6481,Extra data charges
2,0112-QWPNC,Valyermo,One year,49,84.35,"4,059.35",6452,Competitor offered higher download speeds
3,5089-IFSDP,Huntington Beach,Two year,58,109.45,"6,144.55",6424,Product dissatisfaction
4,0406-BPDVR,Visalia,One year,54,101.50,"5,373.10",6405,Lack of affordable download/upload speed
5,4143-HHPMK,Mission Hills,Month-to-month,52,85.35,"4,338.60",6402,Attitude of support person
6,1891-FZYSA,Piercy,Month-to-month,69,89.95,"6,143.15",6363,Network reliability
7,8634-CILSZ,Jamul,One year,69,104.70,"7,220.35",6350,Competitor offered higher download speeds
8,0617-AQNWT,North Hills,Two year,64,47.85,"3,147.50",6347,Competitor made better offer
9,3313-QKNKB,Pebble Beach,One year,59,85.55,"5,084.65",6304,Competitor offered more data
